In [1]:
import heapq
import itertools
from collections import deque

import numpy as np


# ---------- Событие ----------

class Event:

    def __init__(self, env):
        self.env = env
        self._callbacks = []
        self._triggered = False
        self.value = None

    def add_callback(self, callback):
        if self._triggered:
            self.env.schedule(0, callback, event=self)
        else:
            self._callbacks.append(callback)

    def trigger(self, value=None):
        if self._triggered:
            return
        self._triggered = True
        self.value = value
        callbacks, self._callbacks = self._callbacks, []
        for cb in callbacks:
            self.env.schedule(0, cb, event=self)


# ---------- Среда ----------

class Env:
    
    def __init__(self):
        self.now = 0.0
        self._queue = []
        self._counter = itertools.count()

    def schedule(self, delay, callback, **kw):
        heapq.heappush(
            self._queue,
            (self.now + delay, next(self._counter), callback, kw),
        )

    def run(self, until):
        while self._queue:
            time, _, callback, kw = self._queue[0]
            if time > until:
                break
            heapq.heappop(self._queue)
            self.now = time
            callback(**kw)

    def timeout(self, delay):
        event = Event(self)
        self.schedule(delay, lambda **kw: event.trigger())
        return event

    def process(self, gen):
        def step(event=None):
            try:
                next_event = gen.send(event.value if event is not None else None)
            except StopIteration:
                return
            next_event.add_callback(step)

        step()


# ---------- Ресурс ----------

class Resource:
    """Ресурс с ограниченной ёмкостью. request() возвращает Event."""

    def __init__(self, env, capacity=1):
        if capacity < 1:
            raise ValueError("capacity must be >= 1")
        self.env = env
        self.capacity = capacity
        self._free = capacity
        self._waiters = deque()

    def request(self):
        event = Event(self.env)
        if self._free > 0:
            self._free -= 1
            self.env.schedule(0, lambda **kw: event.trigger())
        else:
            self._waiters.append(event)
        return event

    def release(self):
        if self._waiters:
            event = self._waiters.popleft()
            self.env.schedule(0, lambda **kw: event.trigger())
        else:
            self._free += 1


# ---------- Утилиты ----------

def lognormal(rng, median, sigma):
    return median * np.exp(sigma * rng.normal())


# ---------- Модель ----------

def job(env, resources, rng, fail_probs, service_medians, service_sigmas):
    """
    Один процесс, который последовательно использует три ресурса.
    При отказе на любом шаге освобождает текущий ресурс и начинает заново.
    """
    attempt = 1
    while True:
        print(f"\n=== Попытка {attempt} ===")

        for i, (res, p_fail, med, sig) in enumerate(
            zip(resources, fail_probs, service_medians, service_sigmas), start=1
        ):
            print(f"[t={env.now:6.3f}] Запрос ресурса {i}")
            yield res.request()
            print(f"[t={env.now:6.3f}] Ресурс {i} получен")

            service_time = lognormal(rng, median=med, sigma=sig)
            yield env.timeout(service_time)

            if rng.random() < p_fail:
                print(f"[t={env.now:6.3f}] !!! Ресурс {i} отказал")
                res.release()
                attempt += 1
                break  # выходим из for, начинаем while заново
            else:
                print(f"[t={env.now:6.3f}] Ресурс {i} отработал успешно")
                res.release()
        else:
            # for завершился без break — все три ресурса пройдены
            print(f"\n[t={env.now:6.3f}] Все ресурсы успешно пройдены!")
            return

In [36]:
def test(seed):
    rng = np.random.default_rng(seed)
    env = Env()

    # Три ресурса с ёмкостью 1
    res1 = Resource(env, capacity=1)
    res2 = Resource(env, capacity=1)
    res3 = Resource(env, capacity=1)
    resources = [res1, res2, res3]

    # Вероятности отказа на каждом ресурсе
    fail_probs = [0.3, 0.2, 0.1]

    # Параметры логнормального времени обслуживания
    service_medians = [1.0, 1.0, 1.0]
    service_sigmas = [0.2, 0.2, 0.2]

    env.process(job(env, resources, rng, fail_probs,
                    service_medians, service_sigmas))

    env.run(until=100.0)

In [43]:
rnd_seed = np.random.uniform(1,10000)
test(int(rnd_seed))


=== Попытка 1 ===
[t= 0.000] Запрос ресурса 1
[t= 0.000] Ресурс 1 получен
[t= 1.073] !!! Ресурс 1 отказал

=== Попытка 2 ===
[t= 1.073] Запрос ресурса 1
[t= 1.073] Ресурс 1 получен
[t= 2.192] Ресурс 1 отработал успешно
[t= 2.192] Запрос ресурса 2
[t= 2.192] Ресурс 2 получен
[t= 3.821] !!! Ресурс 2 отказал

=== Попытка 3 ===
[t= 3.821] Запрос ресурса 1
[t= 3.821] Ресурс 1 получен
[t= 4.983] Ресурс 1 отработал успешно
[t= 4.983] Запрос ресурса 2
[t= 4.983] Ресурс 2 получен
[t= 5.647] Ресурс 2 отработал успешно
[t= 5.647] Запрос ресурса 3
[t= 5.647] Ресурс 3 получен
[t= 6.675] Ресурс 3 отработал успешно

[t= 6.675] Все ресурсы успешно пройдены!


In [ ]:
"""
A discrete-event simulation environment.

Methods
-------
schedule(delay, callback, **kw)
    Schedule a callback to be executed after a delay.
run(until)
    Execute the simulation loop until the specified time.
timeout(delay)
    Create an event that will be triggered after a delay.
process(gen)
    Helper method to run a generator-based simulation process.
"""

class Env:
    def __init__(self):
        """
        Initialize the simulation environment.

        Sets up the initial time, the internal event queue, and an event counter.
        """
        self.now = 0.0
        self._queue = []
        self._counter = itertools.count()

    def schedule(self, delay, callback, **kw):
        """
        Schedule a callback to be executed after a delay.

        Parameters
        ----------
        delay : float
            The delay before the callback is executed.
        callback : callable
            The callback function.
        **kw : dict
            Additional keyword arguments passed to the callback.
        """
        heapq.heappush(
            self._queue,
            (self.now + delay, next(self._counter), callback, kw),
        )

    def run(self, until):
        """
        Execute the simulation loop until the specified time.

        Parameters
        ----------
        until : float
            The simulation time at which to stop.
        """
        while self._queue:
            time, _, callback, kw = self._queue[0]
            if time > until:
                break
            heapq.heappop(self._queue)
            self.now = time
            callback(**kw)

    def timeout(self, delay):
        """
        Create an event that will be triggered after a delay.

        Parameters
        ----------
        delay : float
            The delay before the event is triggered.

        Returns
        -------
        Event
            The event object.
        """
        event = Event(self)
        self.schedule(delay, lambda **kw: event.trigger())
        return event

    def process(self, gen):
        """
        Helper method to run a generator-based simulation process.

        Parameters
        ----------
        gen : generator
            The simulation generator.
        """
        def step(event=None):
            try:
                next_event = gen.send(event.value if event is not None else None)
            except StopIteration:
                return
            next_event.add_callback(step)

        step()


In [44]:
def test2(seed):
    rng = np.random.default_rng(seed)
    env = Env()

    # Три ресурса, каждый ёмкостью 1
    resources = [Resource(env, capacity=1) for _ in range(3)]

    # Параметры логнормального времени обслуживания на каждом ресурсе
    service_medians = [1.0, 1.0, 1.0]
    service_sigmas = [0.2, 0.2, 0.2]

    # Вероятность успешного завершения раунда (задана заранее)
    p_success_per_round = 0.25

    max_rounds = 10

    def job(env, resources, rng,
            service_medians, service_sigmas,
            p_success, max_rounds):
        """
        До 10 раундов. В каждом раунде три ресурса используются
        строго последовательно. По завершении раунда с вероятностью
        p_success результат считается полученным. Если за 10 раундов
        ни один не оказался успешным — результат всё равно считается
        полученным (по условию).
        """
        for round_num in range(1, max_rounds + 1):
            print(f"\n=== Раунд {round_num} ===")

            # Строго последовательное использование трёх ресурсов
            for i, (res, med, sig) in enumerate(
                zip(resources, service_medians, service_sigmas), start=1
            ):
                print(f"[t={env.now:6.3f}] Запрос ресурса {i}")
                yield res.request()
                print(f"[t={env.now:6.3f}] Ресурс {i} получен")

                service_time = lognormal(rng, median=med, sigma=sig)
                yield env.timeout(service_time)

                res.release()
                print(f"[t={env.now:6.3f}] Ресурс {i} освобождён")

            # Розыгрыш исхода раунда
            if rng.random() < p_success:
                print(f"[t={env.now:6.3f}] "
                      f"Раунд {round_num} успешен — результат получен!")
                return
            else:
                print(f"[t={env.now:6.3f}] "
                      f"Раунд {round_num} неуспешен, переходим к следующему")

        # Прошли все 10 раундов без успеха — по условию результат получен
        print(f"\n[t={env.now:6.3f}] "
              f"10 раундов пройдено — результат получен по умолчанию")

    env.process(job(env, resources, rng,
                    service_medians, service_sigmas,
                    p_success_per_round, max_rounds))

    env.run(until=10_000.0)

In [48]:
rnd_seed = np.random.uniform(1,10000)
test2(int(rnd_seed))


=== Раунд 1 ===
[t= 0.000] Запрос ресурса 1
[t= 0.000] Ресурс 1 получен
[t= 0.988] Ресурс 1 освобождён
[t= 0.988] Запрос ресурса 2
[t= 0.988] Ресурс 2 получен
[t= 2.111] Ресурс 2 освобождён
[t= 2.111] Запрос ресурса 3
[t= 2.111] Ресурс 3 получен
[t= 2.979] Ресурс 3 освобождён
[t= 2.979] Раунд 1 неуспешен, переходим к следующему

=== Раунд 2 ===
[t= 2.979] Запрос ресурса 1
[t= 2.979] Ресурс 1 получен
[t= 4.005] Ресурс 1 освобождён
[t= 4.005] Запрос ресурса 2
[t= 4.005] Ресурс 2 получен
[t= 4.974] Ресурс 2 освобождён
[t= 4.974] Запрос ресурса 3
[t= 4.974] Ресурс 3 получен
[t= 6.189] Ресурс 3 освобождён
[t= 6.189] Раунд 2 успешен — результат получен!
